# **Import Data For Charts**
#### **Data Source:** U.S Energy Information Administration (EIA) Form EIA-923 monthly reports.
#### https://www.eia.gov/electricity/data/eia923/
#### https://docs.catalyst.coop/pudl/en/stable/data_sources/eia923.html

In [2]:
# import
import pandas as pd
import pyarrow
import matplotlib.pyplot as plt

In [3]:
# This data set includes the fuel source and monthly generation data
df_core_eia923_monthly_generation_fuel = pd.read_parquet(
    "s3://pudl.catalyst.coop/nightly/core_eia923__monthly_generation_fuel.parquet",
    dtype_backend="pyarrow",
)

In [4]:
print(df_core_eia923_monthly_generation_fuel.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3340901 entries, 0 to 3340900
Data columns (total 13 columns):
 #   Column                               Dtype                                                       
---  ------                               -----                                                       
 0   report_date                          date32[day][pyarrow]                                        
 1   plant_id_eia                         int32[pyarrow]                                              
 2   energy_source_code                   string[pyarrow]                                             
 3   fuel_type_code_pudl                  dictionary<values=string, indices=int32, ordered=0>[pyarrow]
 4   fuel_type_code_agg                   string[pyarrow]                                             
 5   prime_mover_code                     string[pyarrow]                                             
 6   fuel_consumed_units                  float[pyarrow]       

In [5]:
# This data set includes the nuclear fuel source and monthly generation data
df_core_eia923_monthly_generation_fuel_nuclear = pd.read_parquet(
    "s3://pudl.catalyst.coop/nightly/core_eia923__monthly_generation_fuel_nuclear.parquet",
    dtype_backend="pyarrow",
)

In [6]:
print(df_core_eia923_monthly_generation_fuel_nuclear.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29001 entries, 0 to 29000
Data columns (total 14 columns):
 #   Column                               Non-Null Count  Dtype                                                       
---  ------                               --------------  -----                                                       
 0   plant_id_eia                         29001 non-null  int32[pyarrow]                                              
 1   report_date                          29001 non-null  date32[day][pyarrow]                                        
 2   nuclear_unit_id                      29001 non-null  string[pyarrow]                                             
 3   energy_source_code                   29001 non-null  string[pyarrow]                                             
 4   fuel_type_code_pudl                  29001 non-null  dictionary<values=string, indices=int32, ordered=0>[pyarrow]
 5   fuel_type_code_agg                   29001 non-null  

In [7]:
# This data set includes the plant level data
df_core_eia_entity_plants = pd.read_parquet(
    "s3://pudl.catalyst.coop/nightly/core_eia__entity_plants.parquet",
    dtype_backend="pyarrow",
)

In [8]:
print(df_core_eia_entity_plants.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19082 entries, 0 to 19081
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype                                                       
---  ------          --------------  -----                                                       
 0   plant_id_eia    19082 non-null  int32[pyarrow]                                              
 1   plant_name_eia  18810 non-null  string[pyarrow]                                             
 2   city            17584 non-null  string[pyarrow]                                             
 3   county          18254 non-null  string[pyarrow]                                             
 4   latitude        18233 non-null  float[pyarrow]                                              
 5   longitude       18280 non-null  float[pyarrow]                                              
 6   state           18804 non-null  string[pyarrow]                                             
 7   stre

In [9]:
# Combine the nuclear and non-nuclear fuel data sets into one table

df_generation_combined = pd.concat(
    [
        df_core_eia923_monthly_generation_fuel,
        df_core_eia923_monthly_generation_fuel_nuclear
    ],
    ignore_index=True
)

print(df_generation_combined.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3369902 entries, 0 to 3369901
Data columns (total 14 columns):
 #   Column                               Dtype                                                       
---  ------                               -----                                                       
 0   report_date                          date32[day][pyarrow]                                        
 1   plant_id_eia                         int32[pyarrow]                                              
 2   energy_source_code                   string[pyarrow]                                             
 3   fuel_type_code_pudl                  dictionary<values=string, indices=int32, ordered=0>[pyarrow]
 4   fuel_type_code_agg                   string[pyarrow]                                             
 5   prime_mover_code                     string[pyarrow]                                             
 6   fuel_consumed_units                  float[pyarrow]       

## **Combine Datasets and Export to CSV**

In [10]:
# merge plant metadata into generation table
df_core_eia923_monthly_filtered = df_generation_combined.merge(
    df_core_eia_entity_plants[
        [
            "plant_id_eia",
            "plant_name_eia",
            "city",
            "county",
            "latitude",
            "longitude",
            "state",
            "street_address",
        ]
    ],
    on="plant_id_eia",
    how="left"
)

print(df_core_eia923_monthly_filtered.info())

# export to CSV
df_core_eia923_monthly_filtered.to_csv(
    "/Users/william/Downloads/df_core_eia923_monthly_filtered.csv",
    index=False
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3369902 entries, 0 to 3369901
Data columns (total 21 columns):
 #   Column                               Dtype                                                       
---  ------                               -----                                                       
 0   report_date                          date32[day][pyarrow]                                        
 1   plant_id_eia                         int32[pyarrow]                                              
 2   energy_source_code                   string[pyarrow]                                             
 3   fuel_type_code_pudl                  dictionary<values=string, indices=int32, ordered=0>[pyarrow]
 4   fuel_type_code_agg                   string[pyarrow]                                             
 5   prime_mover_code                     string[pyarrow]                                             
 6   fuel_consumed_units                  float[pyarrow]       

In [11]:
# Create a summary table of fuel types and total generation by fuel type
summary_table = (
    df_core_eia923_monthly_filtered
        .groupby("fuel_type_code_agg")
        .agg(
            count=("fuel_type_code_agg", "size"),
            total_generation_mwh=("net_generation_mwh", "sum")
        )
        .reset_index()
        .sort_values("total_generation_mwh", ascending=False)
)

print(summary_table)

   fuel_type_code_agg   count  total_generation_mwh
0                 COL  209762         35937431552.0
6                  NG  862053         29482323968.0
7                 NUC   28977         19668903936.0
4                 HYC  408572          6534639616.0
14                WND  213461          4515284992.0
13                SUN  458069          1242476288.0
17                WWW   92103           906904832.0
12                RFO   71617           658104384.0
5                 MLG  109840           380093760.0
2                 GEO   17760           373808096.0
10                OTH   88277           302307488.0
8                 OOG   36854           295216672.0
11                 PC   17844           291197088.0
15                WOC    7342           240154240.0
1                 DFO  636599           182928880.0
9                 ORW   46894            58082448.0
16                WOO   52273            29974838.0
3                 HPS   11605          -155842080.0


## **Build Monthly Chart Source in Python for Import to D3 and Export to CSV**

In [14]:
df = df_core_eia923_monthly_filtered.copy()

# map fuel codes to readable names
fuel_map = {
    'COL': 'Coal',
    'NG': 'Natural Gas',
    'NUC': 'Nuclear',
    'SUN': 'Solar',
    'WND': 'Wind',
    'HYC': 'Hydro - Conventional',
    'HPS': 'Hydro Pumped Storage',
    'GEO': 'Geothermal',
    'DFO': 'Distillate Fuel Oil (Diesel)',
    'RFO': 'Residual Fuel Oil',
    'MLG': 'Landfill Gas',
    'OOG': 'Other Gas',
    'WWW': 'Municipal Solid Waste',
    'WOO': 'Wood / Biomass',
    'PC': 'Petroleum Coke',
    'OC': 'Other Coal',
    'ORW': 'Other Renewables',
    'OTH': 'Other / Unknown',
}

df['energy_source_group'] = (
    df['fuel_type_code_agg']
    .astype('string')
    .map(fuel_map)
    .fillna('Other / Unknown')
)

# make sure report_date is datetime
df['report_date'] = pd.to_datetime(df['report_date'])

# aggregate to month + energy source
df_monthly_area = (
    df.groupby(['report_date', 'fuel_type_code_agg','energy_source_group'], as_index=False)
      .agg(net_generation_mwh=('net_generation_mwh', 'sum'))
      .sort_values(['report_date', 'fuel_type_code_agg','energy_source_group'])
)

print(df_monthly_area.head())
print(df_monthly_area.shape)

# export
df_monthly_area.to_csv(
    '/Users/william/Downloads/eia_monthly_generation_by_source.csv',
    index=False
)

  report_date fuel_type_code_agg           energy_source_group  \
0  2001-01-01                COL                          Coal   
1  2001-01-01                DFO  Distillate Fuel Oil (Diesel)   
2  2001-01-01                GEO                    Geothermal   
3  2001-01-01                HPS          Hydro Pumped Storage   
4  2001-01-01                HYC          Hydro - Conventional   

   net_generation_mwh  
0         176287856.0  
1          4003127.75  
2           1223713.0  
3           -588626.0  
4          18822732.0  
(5382, 4)


## **Build State Level Chart Source in Python for Import to D3 and Export to CSV**

In [15]:
# Create a summary table of fuel types and total generation by fuel type by state
df_monthly_state_area = (
    df[
        ['report_date', 'state', 'fuel_type_code_agg', 'energy_source_group','net_generation_mwh']
    ]
    .copy()
)

df_monthly_state_area = (
    df_monthly_state_area
    .dropna(subset=['report_date', 'state', 'fuel_type_code_agg', 'energy_source_group', 'net_generation_mwh'])
    .groupby(['report_date', 'state', 'fuel_type_code_agg', 'energy_source_group'], as_index=False)
    .agg(net_generation_mwh=('net_generation_mwh', 'sum'))
    .sort_values(['state', 'report_date', 'fuel_type_code_agg', 'energy_source_group'])
)

print(df_monthly_state_area.info())
print(df_monthly_state_area.head())

df_monthly_state_area.to_csv(
    '/Users/william/Downloads/eia_monthly_generation_by_state_by_source.csv',
    index=False
)

<class 'pandas.core.frame.DataFrame'>
Index: 179572 entries, 0 to 179571
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype          
---  ------               --------------   -----          
 0   report_date          179572 non-null  datetime64[ns] 
 1   state                179572 non-null  string[pyarrow]
 2   fuel_type_code_agg   179572 non-null  string[pyarrow]
 3   energy_source_group  179572 non-null  object         
 4   net_generation_mwh   179572 non-null  float[pyarrow] 
dtypes: datetime64[ns](1), float[pyarrow](1), object(1), string[pyarrow](2)
memory usage: 7.0+ MB
None
  report_date state fuel_type_code_agg           energy_source_group  \
0  2001-01-01    AK                COL                          Coal   
1  2001-01-01    AK                DFO  Distillate Fuel Oil (Diesel)   
2  2001-01-01    AK                HYC          Hydro - Conventional   
3  2001-01-01    AK                 NG                   Natural Gas   
4  2001-01-01    AK

In [16]:
# Reuse mappings and group fuel sources into simplified categories

# Detailed / aggregated fuel code -> simplified chart group
FUEL_AGG_TO_GROUP = {
    "SUN": "Solar",
    "WND": "Wind",
    "HYC": "Hydro",
    "HPS": "Hydro",
    "NUC": "Nuclear",
    "COL": "Coal",
    "NG": "Natural Gas",
    "DFO": "Other",
    "RFO": "Other",
    "MLG": "Other",
    "OOG": "Other",
    "WWW": "Other",
    "WOO": "Other",
    "PC": "Other",
    "OC": "Other",
    "ORW": "Other",
    "OTH": "Other",
    "GEO": "Other",
}

# EIA energy_source_code_1 -> simplified chart group
EIA_ENERGY_SOURCE_TO_GROUP = {
    "SUN": "Solar",
    "WND": "Wind",
    "HYC": "Hydro",
    "HPS": "Hydro",
    "WAT": "Hydro",
    "NUC": "Nuclear",
    "COL": "Coal",
    "BIT": "Coal",
    "SUB": "Coal",
    "LIG": "Coal",
    "WC": "Coal",
    "RC": "Coal",
    "NG": "Natural Gas",
    "OG": "Natural Gas",
    "BFG": "Natural Gas",
    "PG": "Natural Gas",
    "SGC": "Natural Gas",
    "SGP": "Natural Gas",
    "DFO": "Other",
    "RFO": "Other",
    "PC": "Other",
    "MLG": "Other",
    "OOG": "Other",
    "WWW": "Other",
    "WOO": "Other",
    "GEO": "Other",
    "PUR": "Other",
    "AB": "Other",
    "MSB": "Other",
    "OBS": "Other",
    "WH": "Other",
    "OTH": "Other",
    "ORW": "Other",
}

def map_generation_group(code):
    """Map fuel_type_code_agg to simplified chart group."""
    if pd.isna(code):
        return "Other"
    return FUEL_AGG_TO_GROUP.get(str(code), "Other")

def map_eia_group(code):
    """Map EIA-860 energy_source_code_1 to simplified chart group."""
    if pd.isna(code):
        return "Other"
    return EIA_ENERGY_SOURCE_TO_GROUP.get(str(code), "Other")

def choose_top_category(df, category_col, weight_col, out_col_name):
    """
    For each plant-year choose the category with the largest weight.
    Ties are broken alphabetically for reproducibility.
    """
    tmp = (
        df.groupby(["plant_id_eia", "report_year", category_col], dropna=False, as_index=False)[weight_col]
          .sum()
          .rename(columns={weight_col: "weight"})
    )

    # stable sort: largest weight first, then category asc
    tmp = tmp.sort_values(
        ["plant_id_eia", "report_year", "weight", category_col],
        ascending=[True, True, False, True],
        na_position="last"
    )

    top = (
        tmp.groupby(["plant_id_eia", "report_year"], as_index=False)
           .first()
           .rename(columns={category_col: out_col_name, "weight": f"{out_col_name}_weight"})
    )
    return top


# Load the annual EIA-860 generator table

# This table provides generator-level year-end capacity and EIA fuel coding.
# PUDL docs show out_eia__yearly_generators_by_ownership as an annual table
# with capacity_eoy_mw and energy_source_code_1.

df_eia860_yearly_gen = pd.read_parquet(
    "s3://pudl.catalyst.coop/nightly/out_eia__yearly_generators_by_ownership.parquet",
    dtype_backend="pyarrow",
)

# Convert report_date -> year and keep only 2001-2024
df_eia860_yearly_gen["report_date"] = pd.to_datetime(df_eia860_yearly_gen["report_date"])
df_eia860_yearly_gen["report_year"] = df_eia860_yearly_gen["report_date"].dt.year
df_eia860_yearly_gen = df_eia860_yearly_gen.loc[
    df_eia860_yearly_gen["report_year"].between(2001, 2024)
].copy()

# Pick the best available year-end capacity column
capacity_col = None
for col in ["capacity_eoy_mw", "capacity_mw"]:
    if col in df_eia860_yearly_gen.columns:
        capacity_col = col
        break

if capacity_col is None:
    raise ValueError("Could not find capacity_eoy_mw or capacity_mw in out_eia__yearly_generators_by_ownership.")

# Optional sanity check
needed_cols = ["plant_id_eia", "generator_id", "report_date", "report_year", "energy_source_code_1", capacity_col]
missing = [c for c in needed_cols if c not in df_eia860_yearly_gen.columns]
if missing:
    raise ValueError(f"Missing required columns in annual generator table: {missing}")


# Build annual plant-year generation totals from existing monthly table

df_gen = df_core_eia923_monthly_filtered.copy()
df_gen["report_date"] = pd.to_datetime(df_gen["report_date"])
df_gen["report_year"] = df_gen["report_date"].dt.year

df_gen = df_gen.loc[df_gen["report_year"].between(2001, 2024)].copy()

# Annual generation by plant-year-fuel
plant_year_fuel = (
    df_gen.groupby(
        [
            "plant_id_eia",
            "report_year",
            "plant_name_eia",
            "latitude",
            "longitude",
            "state",
            "fuel_type_code_agg",
        ],
        dropna=False,
        as_index=False,
    )["net_generation_mwh"]
    .sum()
)

plant_year_fuel["energy_source_group"] = plant_year_fuel["fuel_type_code_agg"].map(map_generation_group)

# Total annual generation by plant-year
plant_year_total_gen = (
    plant_year_fuel.groupby(
        ["plant_id_eia", "report_year", "plant_name_eia", "latitude", "longitude", "state"],
        dropna=False,
        as_index=False,
    )["net_generation_mwh"]
    .sum()
    .rename(columns={"net_generation_mwh": "total_generation_mwh"})
)

# Dominant detailed fuel_type_code_agg by annual generation
dominant_fuel_code = choose_top_category(
    plant_year_fuel,
    category_col="fuel_type_code_agg",
    weight_col="net_generation_mwh",
    out_col_name="fuel_type_code_agg"
)

# Dominant simplified energy group by annual generation
dominant_energy_group = choose_top_category(
    plant_year_fuel,
    category_col="energy_source_group",
    weight_col="net_generation_mwh",
    out_col_name="dominant_energy_group"
)

# Build year-end plant-year metadata from annual EIA-860 generators

# Keep a plant-year-level location/name fallback from generation table
plant_year_location = (
    plant_year_total_gen[
        ["plant_id_eia", "report_year", "plant_name_eia", "latitude", "longitude", "state"]
    ]
    .drop_duplicates()
)

# Installed capacity at year end (sum across generators)
plant_year_capacity = (
    df_eia860_yearly_gen.groupby(["plant_id_eia", "report_year"], as_index=False)[capacity_col]
    .sum()
    .rename(columns={capacity_col: "installed_capacity_mw"})
)

# Primary technology nameplate at year end
if "technology_description" in df_eia860_yearly_gen.columns:
    plant_year_technology = choose_top_category(
        df_eia860_yearly_gen.assign(
            capacity_weight=df_eia860_yearly_gen[capacity_col].fillna(0)
        ),
        category_col="technology_description",
        weight_col="capacity_weight",
        out_col_name="primary_technology_nameplate",
    )[["plant_id_eia", "report_year", "primary_technology_nameplate"]]
else:
    plant_year_technology = plant_year_capacity[["plant_id_eia", "report_year"]].copy()
    plant_year_technology["primary_technology_nameplate"] = pd.NA

# Official EIA year-end classification proxy:
# pick plant-year winner based on energy_source_code_1 weighted by year-end capacity
plant_year_eia_code = choose_top_category(
    df_eia860_yearly_gen.assign(
        capacity_weight=df_eia860_yearly_gen[capacity_col].fillna(0)
    ),
    category_col="energy_source_code_1",
    weight_col="capacity_weight",
    out_col_name="official_plant_classification_eia_code",
)

plant_year_eia_code["official_plant_classification_eia_group"] = (
    plant_year_eia_code["official_plant_classification_eia_code"].map(map_eia_group)
)

plant_year_eia_code = plant_year_eia_code[
    [
        "plant_id_eia",
        "report_year",
        "official_plant_classification_eia_code",
        "official_plant_classification_eia_group",
    ]
]

# Nameplate fuel group proxy from energy_source_code_1, also capacity-weighted
plant_year_nameplate_group = choose_top_category(
    df_eia860_yearly_gen.assign(
        energy_source_group_nameplate=df_eia860_yearly_gen["energy_source_code_1"].map(map_eia_group),
        capacity_weight=df_eia860_yearly_gen[capacity_col].fillna(0)
    ),
    category_col="energy_source_group_nameplate",
    weight_col="capacity_weight",
    out_col_name="nameplate_energy_group",
)[["plant_id_eia", "report_year", "nameplate_energy_group"]]

# Assemble final annual plant-year table
plant_year_table = (
    plant_year_total_gen
    .merge(dominant_fuel_code, on=["plant_id_eia", "report_year"], how="left")
    .merge(dominant_energy_group, on=["plant_id_eia", "report_year"], how="left")
    .merge(plant_year_capacity, on=["plant_id_eia", "report_year"], how="left")
    .merge(plant_year_technology, on=["plant_id_eia", "report_year"], how="left")
    .merge(plant_year_eia_code, on=["plant_id_eia", "report_year"], how="left")
    .merge(plant_year_nameplate_group, on=["plant_id_eia", "report_year"], how="left")
)

# Use generation-based simplified group as requested output column
# (this mirrors the chart category for the selected year)
plant_year_table["energy_source_group"] = plant_year_table["dominant_energy_group"]

# Difference flags across annual classification methods
plant_year_table["diff_generation_vs_nameplate_group"] = (
    plant_year_table["dominant_energy_group"] != plant_year_table["nameplate_energy_group"]
)

plant_year_table["diff_generation_vs_eia_group"] = (
    plant_year_table["dominant_energy_group"] != plant_year_table["official_plant_classification_eia_group"]
)

plant_year_table["diff_nameplate_vs_eia_group"] = (
    plant_year_table["nameplate_energy_group"] != plant_year_table["official_plant_classification_eia_group"]
)

plant_year_table["any_classification_difference"] = (
    plant_year_table[
        [
            "diff_generation_vs_nameplate_group",
            "diff_generation_vs_eia_group",
            "diff_nameplate_vs_eia_group",
        ]
    ]
    .fillna(False)
    .any(axis=1)
)

# Optional count of distinct non-null simplified groups across methods
def count_distinct_nonnull(row):
    vals = {
        row.get("dominant_energy_group"),
        row.get("nameplate_energy_group"),
        row.get("official_plant_classification_eia_group"),
    }
    vals = {v for v in vals if pd.notna(v)}
    return len(vals)

plant_year_table["n_distinct_classification_groups"] = plant_year_table.apply(
    count_distinct_nonnull, axis=1
)

# Final column order

final_cols = [
    "plant_id_eia",
    "report_year",
    "plant_name_eia",
    "latitude",
    "longitude",
    "state",
    "fuel_type_code_agg",                          # dominant detailed generation code in selected year
    "energy_source_group",                         # simplified group for selected year
    "installed_capacity_mw",                       # year-end installed capacity
    "primary_technology_nameplate",                # year-end top technology_description by capacity
    "official_plant_classification_eia_code",      # year-end EIA proxy code (energy_source_code_1 winner)
    "official_plant_classification_eia_group",     # simplified version of EIA proxy code
    "dominant_energy_group",                       # annual generation winner
    "total_generation_mwh",                        # annual generation total
    "nameplate_energy_group",                      # year-end nameplate simplified group
    "diff_generation_vs_nameplate_group",
    "diff_generation_vs_eia_group",
    "diff_nameplate_vs_eia_group",
    "any_classification_difference",
    "n_distinct_classification_groups",
]

plant_year_table = plant_year_table[final_cols].sort_values(
    ["report_year", "state", "plant_name_eia", "plant_id_eia"]
)

print(plant_year_table.info())
print(plant_year_table.head(10))

# Export results to csv

output_path = "/Users/william/Downloads/eia_plant_year_dominant_source_2001_2024.csv"
plant_year_table.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

<class 'pandas.core.frame.DataFrame'>
Index: 180457 entries, 90533 to 63973
Data columns (total 20 columns):
 #   Column                                   Non-Null Count   Dtype          
---  ------                                   --------------   -----          
 0   plant_id_eia                             180457 non-null  int32[pyarrow] 
 1   report_year                              180457 non-null  int32          
 2   plant_name_eia                           179628 non-null  string[pyarrow]
 3   latitude                                 178953 non-null  float[pyarrow] 
 4   longitude                                179281 non-null  float[pyarrow] 
 5   state                                    179541 non-null  string[pyarrow]
 6   fuel_type_code_agg                       180457 non-null  string[pyarrow]
 7   energy_source_group                      180457 non-null  object         
 8   installed_capacity_mw                    179533 non-null  float[pyarrow] 
 9   primary_technolog